H0: Применение кросс валидации временных рядов с разбиением на фолды не влияет на качество модели https://habrastorage.org/files/f5c/7cd/b39/f5c7cdb39ccd4ba68378ca232d20d864.png

H1: Применение кросс валидации временных рядов с разбиением на фолды положительно влияет на качество модели

Уровень значимости: 5%

Статистика: Тесты для связанных выборок. t-тест Стюдента для зависимых выборок / критерий знаков Вилкоксона

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
from scipy.stats import wilcoxon

In [2]:
def prepare_data(cros_val_path, standard_path):

    cross_val_df = pd.read_csv(cros_val_path, index_col=0).sort_index()
    standard_df = pd.read_csv(standard_path, index_col=0).sort_index()
    cross_val_df.drop('mape', axis=1, inplace=True)
    standard_df.drop('mape', axis=1, inplace=True)

    df_ = pd.concat(
        (
            cross_val_df[['date', 'product', 'product_type', 'federal_okrug', 'price']].rename(columns={'price':'cros_val_price'}),
            standard_df['price'].rename('standard_price')
        ),
        axis=1
    )
    df_ = df_.dropna(axis=0)

    df = pd.read_csv('true_data.csv', index_col=0)
    df['product'], df['product_type'], df['federal_okrug'] = zip(*df['ID'].apply(lambda x: x.split('|')))
    df = df.drop('ID', axis=1)
    df = df.rename(columns={'ds':'date'})
    df = pd.merge(df_, df, how='left', on=['date', 'product', 'product_type', 'federal_okrug'])
    df = df.dropna(axis=0)
    df['mape_cros_val'] = np.abs((df['y'] - df['cros_val_price']) / df['y']) * 100
    df['mape_standard'] = np.abs((df['y'] - df['standard_price']) / df['y']) * 100
    return df

df = prepare_data('cross_val_preds_3_mons.csv', 'standard_preds_3_mons.csv')

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df['mape_cros_val'], label='кросс валидация', bins=50, alpha=0.5)
plt.hist(df['mape_standard'], label='без кросс валидации', bins=50, alpha=0.5)
plt.legend()
plt.show()

Распределение ненормальное, поэтому используем критерий знаков Вилкоксона.

Нулевая гипотеза критерия знаков, предполагает, что выборки пришли из одного распределения.

In [3]:
print(f"среднее: {round(np.mean(df['mape_cros_val']), 2)} медиана: {round(np.median(df['mape_cros_val']), 2)} - выборка с k-fold кросс-валидацией")
print(f"среднее: {round(np.mean(df['mape_standard']), 2)} медиана: {round(np.median(df['mape_standard']), 2)} - выборка с обычной валидацией")
pvalue = wilcoxon(df['mape_cros_val'], df['mape_standard']).pvalue
print(pvalue, 'Выборки из разных распределений -', pvalue < 0.05)

среднее: 7.49 медиана: 5.61 - выборка с k-fold кросс-валидацией
среднее: 7.83 медиана: 5.64 - выборка с обычной валидацией
0.003989521860489568 Выборки из разных распределений - True


In [ ]:
df = prepare_data('cross_val_preds_1_year.csv', 'standard_preds_1_year.csv')
plt.figure(figsize=(10, 6))
plt.hist(df['mape_cros_val'], label='кросс валидация', bins=50, alpha=0.5)
plt.hist(df['mape_standard'], label='без кросс валидации', bins=50, alpha=0.5)
plt.legend()
plt.show()

In [4]:
print(f"среднее: {round(np.mean(df['mape_cros_val']), 2)} медиана: {round(np.median(df['mape_cros_val']), 2)} - выборка с k-fold кросс-валидацией")
print(f"среднее: {round(np.mean(df['mape_standard']), 2)} медиана: {round(np.median(df['mape_standard']), 2)} - выборка с обычной валидацией")
pvalue = wilcoxon(df['mape_cros_val'], df['mape_standard']).pvalue
print(pvalue, 'Выборки из разных распределений -', pvalue < 0.05)

среднее: 7.49 медиана: 5.61 - выборка с k-fold кросс-валидацией
среднее: 7.83 медиана: 5.64 - выборка с обычной валидацией
0.003989521860489568 Выборки из разных распределений - True


In [ ]:
df_history = pd.read_csv('true_data.csv', index_col=0)
df_history['product'], df_history['product_type'], df_history['federal_okrug'] = zip(*df_history['ID'].apply(lambda x: x.split('|')))
df_history = df_history.rename(columns={'ds':'date'})
df_history = pd.merge(df_history, df, how='left', on=['date', 'product', 'product_type', 'federal_okrug'])
df_history['date'] = pd.to_datetime(df_history['date'])

In [ ]:
import matplotlib.dates as mdates

for id in df_history['ID'].unique():
    df_ = df_history[df_history['ID'] == id]
    plt.figure(figsize=(18, 6))
    plt.plot(df_['date'], df_['y_x'], color='blue', label='реальные данные' )
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=30))
    plt.gcf().autofmt_xdate()
    plt.plot(df_['date'], df_['cros_val_price'], color='red', linestyle='--', label='кросс валидация', alpha=0.5)
    plt.plot(df_['date'], df_['standard_price'], color='green', linestyle=':', label='без кросс валидации',  alpha=0.5)
    plt.legend()
    plt.show()